# ////////////////////////////////////////////////////////////////mycode///////////////////////////////////////////////////////////////

In [ ]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [3]:
 pip install sounddevice

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.7 MB/s  0:00:005.5 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install vosk

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 1.4 MB/s  0:00:03m 1.3 MB/s eta 0:00:010m
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22512 sha256=e5c9a768c7a02ace72516784fd6ee2cb9d3ce1abff021fc0ee805bcab19bc089
  Stored in directory: /Users/alizeaarif/Library/Caches/pip/wheels/ed/9d/54/8e040c593adecbf10de5a6263add15cfefff5a033e33fda8c6
Successfully built srt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [vosk]
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.7 MB/s  0:00:011.8 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]
Note: you may need to restart the kernel to use updated packages.


In [1]:
#importing libraries
import os
import json
import wave
import numpy as np
import soundfile as sf
from vosk import Model, KaldiRecognizer
from jiwer import wer as calculate_wer

In [2]:
import sounddevice as sd
RECORDING_AVAILABLE = True

In [3]:
class AirportASR:
    #initializing the ASR system with models from a specified path
    def __init__(self, models_path='models'):
        self.models_path = models_path
        self.models = {}

        #directory names for different language models
        self.model_dirs = {
            'en': 'vosk-model-en-us-0.22',
            'es': 'vosk-model-small-es-0.42',
            'it': 'vosk-model-small-it-0.22'
        }

        #loading the models when the system is initialized
        print("Initializing Airport ASR System...")
        self._load_models()  

    #loading the models for all specified languages
    def _load_models(self):
        for lang, model_dir in self.model_dirs.items():
            model_path = os.path.join(self.models_path, model_dir)
            try:
                print(f"Loading {lang.upper()} model from {model_path}...")
                #load the language model
                self.models[lang] = Model(model_path)  
                print(f"✓ {lang.upper()} model loaded successfully")
            except Exception as e:
                print(f"✗ Error loading {lang.upper()} model: {e}")

    #records audio and savess it to a file
    def record_audio(self, filename, duration=5, sample_rate=16000):
        if not RECORDING_AVAILABLE:
            print("Recording not available. Install sounddevice: pip install sounddevice")
            return None

        print(f"\n🎤 Recording for {duration} seconds...")
        print("Speak now!")

        recording = sd.rec(int(duration * sample_rate),
                           samplerate=sample_rate,
                           channels=1,
                           dtype='float32')
        #waits for the recording to complete
        sd.wait()  

        print("✓ Recording complete!")

        #saves the recorded audio to the specified file
        sf.write(filename, recording, sample_rate, subtype='PCM_16')
        print(f"✓ Saved to: {filename}")

        return filename

    #applies a simple lowpass filter to audio data
    def simple_lowpass_filter(self, data, sample_rate, cutoff=3500):
        window_size = int(sample_rate / cutoff)
        if window_size < 3:
            window_size = 3
        if window_size % 2 == 0:
            window_size += 1

        #create the filter kernel and apply it to the data
        kernel = np.ones(window_size) / window_size
        filtered = np.convolve(data, kernel, mode='same')

        return filtered

    #preprocesses the audio for better transcription accuracy
    def preprocess_audio_enhanced(self, audio_path, output_path=None, aggressive=False):
        print(f"Preprocessing: {os.path.basename(audio_path)}")

        audio_data, sample_rate = sf.read(audio_path)

        if len(audio_data.shape) > 1:
            #convert stereo to mono if necessary
            audio_data = np.mean(audio_data, axis=1)  

        audio_float = audio_data.astype(np.float32)

        #normalize audio to have zero mean
        audio_float = audio_float - np.mean(audio_float)

        max_val = np.max(np.abs(audio_float))
        if max_val > 0:
            #scales the audio
            audio_float = audio_float / max_val * 0.95  

        #calculates root mean square
        rms = np.sqrt(np.mean(audio_float**2))  
        #target RMS value for normalization
        target_rms = 0.15  
        if rms > 0:
            gain_factor = target_rms / rms
             #clamping gain to avoid extreme values
            gain_factor = np.clip(gain_factor, 0.5, 3.0) 
            audio_float = audio_float * gain_factor

        #applying low pass filter based on aggressiveness
        cutoff = 3800 if not aggressive else 3200
        audio_filtered = self.simple_lowpass_filter(audio_float, sample_rate, cutoff=cutoff)

        #aggressive noise reduction if specified
        if aggressive:
            sorted_audio = np.sort(np.abs(audio_filtered))
            #calculate noise threshold
            noise_threshold = sorted_audio[int(len(sorted_audio) * 0.1)]  
            noise_threshold = max(noise_threshold, 0.01)

            mask = np.abs(audio_filtered) < noise_threshold
            #reduce noise
            audio_filtered[mask] *= 0.3  
        else:
            noise_threshold = 0.015
             #mild noise reduction
            audio_filtered[np.abs(audio_filtered) < noise_threshold] *= 0.5  
            
        max_val = np.max(np.abs(audio_filtered))
        if max_val > 0:
             #normalize again after filtering
            audio_filtered = audio_filtered / max_val * 0.9 

        #cliping values to [-1, 1]
        audio_filtered = np.clip(audio_filtered, -1.0, 1.0)  

        if output_path is None:
            base, ext = os.path.splitext(audio_path)
            output_path = f"{base}_preprocessed{ext}"

        #saves the preprocessed audio
        sf.write(output_path, audio_filtered, sample_rate, subtype='PCM_16')
        print(f"  → Saved: {output_path}")

        return output_path

    #transcribes the given audio file to text
    def transcribe(self, audio_path, language='en', preprocess=True, aggressive=False):
        if language not in self.models:
            raise ValueError(f"Language '{language}' not supported or model not loaded")

        if preprocess:
            audio_path = self.preprocess_audio_enhanced(audio_path, aggressive=aggressive)

        wf = wave.open(audio_path, "rb")

        if wf.getnchannels() != 1:
            wf.close()
            audio_data, sample_rate = sf.read(audio_path)
            if len(audio_data.shape) > 1:
                audio_data = np.mean(audio_data, axis=1)  
            temp_path = audio_path.replace('.wav', '_mono.wav')
            sf.write(temp_path, audio_data, sample_rate, subtype='PCM_16')
            wf = wave.open(temp_path, "rb")

        rec = KaldiRecognizer(self.models[language], wf.getframerate())
        rec.SetWords(True)

        transcription = ""
        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break
            if rec.AcceptWaveform(data):
                result = json.loads(rec.Result())
                if 'text' in result:
                    transcription += result['text'] + " "

        #getting final result from Kaldi recognizer
        final_result = json.loads(rec.FinalResult())
        if 'text' in final_result:
            transcription += final_result['text']

        wf.close()

        return transcription.strip()

    #calculates the word error rate between reference and hypothesis
    def calculate_wer(self, reference, hypothesis):
        if not reference.strip():
            return 100.0 if hypothesis.strip() else 0.0

        error_rate = calculate_wer(reference.lower(), hypothesis.lower())
        return round(error_rate * 100, 2)
        

    #evaluate the test set and calculate WER for each file
    def evaluate_test_set(self, audio_dir, transcriptions_dict, language='en', aggressive=False):
        results = []

        print(f"\n{'='*70}")
        print(f"Evaluating {language.upper()} Test Set")
        print(f"{'='*70}")

        for filename, reference in transcriptions_dict.items():
            audio_path = os.path.join(audio_dir, filename)

            if not os.path.exists(audio_path):
                print(f"⚠ File not found: {filename}")
                continue

            print(f"\nProcessing: {filename}")
            print(f"Reference: {reference}")

            hypothesis = self.transcribe(audio_path, language, preprocess=True, aggressive=aggressive)
            print(f"Hypothesis: {hypothesis}")

            wer_score = self.calculate_wer(reference, hypothesis)
            print(f"WER: {wer_score}%")

            results.append({
                'Language': language.upper(),
                'File': filename,
                'Reference': reference,
                'Hypothesis': hypothesis,
                'WER': f"{wer_score}%"
            })

        return results

In [4]:

#recording custom sentences and test the ASR system with them
def record_custom_sentences(asr):
    print("\n" + "="*70)
    print("CUSTOM RECORDING SESSION")
    print("="*70)

    #list to store results of custom recordings
    custom_results = []  

    #list of custom sentences to be recorded and transcribed
    sentences = [
        ("I need help finding my gate", "en"),
        ("Where is the nearest bathroom", "en")
    ]

    #loop through each sentence for recording
    for i, (reference, lang) in enumerate(sentences, 1):
        # Prompt user to record each sentence
        print(f"\n--- Recording {i}/2 ---")
        print(f"Please say: '{reference}'")
        input("Press ENTER when ready to record...")
        
        #define filename for the recording
        filename = f"your_sentence{i}.wav"  
        recorded_file = asr.record_audio(filename, duration=5, sample_rate=16000)

        if recorded_file:
            #transcribe the recorded audio and calculate wer
            print("\nTranscribing...")
            hypothesis = asr.transcribe(recorded_file, lang, preprocess=True, aggressive=True)
            wer_score = asr.calculate_wer(reference, hypothesis)

            #print reference, transcribed result, and WER score
            print(f"\n Reference:  {reference}")
            print(f" Transcribed: {hypothesis}")
            print(f" WER: {wer_score}%")

            #aapend the result to custom_results list
            custom_results.append({
                'Language': lang.upper(),
                'File': filename,
                'Reference': reference,
                'Hypothesis': hypothesis,
                'WER': f"{wer_score}%"
            })

    #return the results for further processing
    return custom_results  

In [5]:
#main function to run the evaluation
def main():

    #initialize the Airport ASR system with models
    asr = AirportASR(models_path='models')

    #defining expected transcriptions for test files in different languages
    english_transcriptions = {
        'checkin.wav': 'where is the check-in desk',
        'parents.wav': 'i have lost my parents',
        'suitcase.wav': 'please i have lost my suitcase',
        'what_time.wav': 'what time is my plane',
        'where.wav': 'where are the restaurants and shops',
    }

    italian_transcriptions = {
        'checkin_it.wav': "dove e' il bancone",
        'parents_it.wav': 'ho perso i miei genitori',
        'suitcase_it.wav': 'per favore ho perso la mia valigia',
        'what_time_it.wav': "a che ora e' il mio aereo",
        'where_it.wav': 'dove sono i ristoranti e i negozi',
    }

    spanish_transcriptions = {
        'checkin_es.wav': 'dónde están los mostradores',
        'parents_es.wav': 'he perdido a mis padres',
        'suitcase_es.wav': 'por favor he perdido mi maleta',
        'what_time_es.wav': 'a qué hora es mi avión',
        'where_es.wav': 'dónde están los restaurantes y las tiendas',
    }

    #list to store results from all evaluations
    all_results = []  

    #evaluate test sets for each language and extend results list
    if 'en' in asr.models:
        results_en = asr.evaluate_test_set('Ex4_audio_files/EN', english_transcriptions, 'en', aggressive=True)
        all_results.extend(results_en)

    if 'es' in asr.models:
        results_es = asr.evaluate_test_set('Ex4_audio_files/ES', spanish_transcriptions, 'es')
        all_results.extend(results_es)

    if 'it' in asr.models:
        results_it = asr.evaluate_test_set('Ex4_audio_files/IT', italian_transcriptions, 'it')
        all_results.extend(results_it)

    #ask if the user wants to record custom sentences
    print("\n" + "="*70)
    print("Would you like to record custom sentences? (y/n)")
    response = input().lower().strip()

    if response == 'y':
        custom_results = record_custom_sentences(asr)
        all_results.extend(custom_results)
    else:
        print("Skipping custom recordings.")

    #printing final results table with wer
    print(f"\n{'='*70}")
    print("FINAL RESULTS TABLE")
    print(f"{'='*70}")
    print(f"{'Language':<12} {'File':<30} {'WER':<10}")
    print(f"{'-'*70}")

    for result in all_results:
        print(f"{result['Language']:<12} {result['File']:<30} {result['WER']:<10}")

    #print average WER by language
    print(f"\n{'='*70}")
    print("AVERAGE WER BY LANGUAGE")
    print(f"{'='*70}")

    #calculate and display the average WER for each language, and compare against threshold
    for lang in ['EN', 'ES', 'IT']:
        lang_results = [r for r in all_results if r['Language'] == lang]
        if lang_results:
            avg_wer = np.mean([float(r['WER'].rstrip('%')) for r in lang_results])
            threshold = 25 if lang == 'EN' else 35  
             #passor fail based on threshold
            status = "✓ PASS" if avg_wer < threshold else "✗ FAIL" 
            print(f"{lang}: {avg_wer:.2f}% (threshold: <{threshold}%) {status}")

    #return the collected results
    return all_results  

In [6]:
#execute
if __name__ == "__main__":
    results = main()

Initializing Airport ASR System...
Loading EN model from models/vosk-model-en-us-0.22...


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=6
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:11:12:13:14:15
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from models/vosk-model-en-us-0.22/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from models/vosk-model-en-us-0.22/graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:294) Loading words from models/vosk-model-en-us-0.22/graph/words.txt
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo models/vosk-model-en-us-0.22/graph/phones/word_boundary.int
LOG (VoskAPI:ReadDataFiles():mod

✓ EN model loaded successfully
Loading ES model from models/vosk-model-small-es-0.42...


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=11 max-active=4000 lattice-beam=4
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from models/vosk-model-small-es-0.42/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from models/vosk-model-small-es-0.42/graph/HCLr.fst models/vosk-model-small-es-0.42/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo models/vosk-model-small-es-0.42/graph/phones/word_boundary.int
LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 l

✓ ES model loaded successfully
Loading IT model from models/vosk-model-small-it-0.22...


...


CUSTOM RECORDING SESSION

--- Recording 1/2 ---
Please say: 'I need help finding my gate'


Press ENTER when ready to record... 



🎤 Recording for 5 seconds...
Speak now!
✓ Recording complete!
✓ Saved to: your_sentence1.wav

Transcribing...
Preprocessing: your_sentence1.wav
  → Saved: your_sentence1_preprocessed.wav

 Reference:  I need help finding my gate
 Transcribed: i need help finding my gate
 WER: 0.0%

--- Recording 2/2 ---
Please say: 'Where is the nearest bathroom'


Press ENTER when ready to record... 



🎤 Recording for 5 seconds...
Speak now!
✓ Recording complete!
✓ Saved to: your_sentence2.wav

Transcribing...
Preprocessing: your_sentence2.wav
  → Saved: your_sentence2_preprocessed.wav

 Reference:  Where is the nearest bathroom
 Transcribed: where is the nearest bathroom
 WER: 0.0%

FINAL RESULTS TABLE
Language     File                           WER       
----------------------------------------------------------------------
EN           checkin.wav                    40.0%     
EN           parents.wav                    20.0%     
EN           suitcase.wav                   0.0%      
EN           what_time.wav                  20.0%     
EN           where.wav                      0.0%      
ES           checkin_es.wav                 0.0%      
ES           parents_es.wav                 0.0%      
ES           suitcase_es.wav                0.0%      
ES           what_time_es.wav               83.33%    
ES           where_es.wav                   14.29%    
IT           che

# ////////////////////////////////////////////////////////////////mycode///////////////////////////////////////////////////////////////
